# 🎤 親子対話：VAD — POC

Voice Activity Detection = 音声区間検出。「使う部分と捨てる部分を分ける」。

📖 対応記事: `親子対話：VAD`

In [ ]:
!pip install -q silero-vad torch torchaudio

In [ ]:
# サンプル音声生成（無音＋発話＋無音＋発話）
import torch
import torchaudio
import numpy as np

sr = 16000
dur = 1.0
t = np.linspace(0, dur, int(sr*dur), endpoint=False)

# 発話風: 440Hz +  harmonics
voice1 = np.sin(2*np.pi*200*t) + 0.5*np.sin(2*np.pi*400*t) + 0.3*np.sin(2*np.pi*600*t)
voice1 *= np.exp(-3*t)  # decay
voice2 = np.sin(2*np.pi*300*t) + 0.4*np.sin(2*np.pi*600*t)
voice2 *= np.exp(-2*t)

silence = np.zeros(int(sr * 0.5))
noise = np.random.randn(int(sr * 0.3)) * 0.02

audio = np.concatenate([silence, voice1, silence, noise, voice2, silence])
audio_tensor = torch.FloatTensor(audio).unsqueeze(0)

import matplotlib.pyplot as plt
plt.figure(figsize=(12, 3))
plt.plot(audio)
plt.title('Input: Silence → Voice → Silence → Noise → Voice → Silence')
plt.show()

In [ ]:
# Silero VAD 推論
model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad')
(get_speech_timestamps, _, _, _, _) = utils

speech_timestamps = get_speech_timestamps(audio_tensor, model, sampling_rate=sr)

print(f"🎯 Detected {len(speech_timestamps)} speech segments:")
for seg in speech_timestamps:
    start = seg['start'] / sr
    end = seg['end'] / sr
    print(f"   {start:.2f}s → {end:.2f}s (duration: {end-start:.2f}s)")

In [ ]:
# 可視化: 青＝音声区間
import matplotlib.patches as mpatches

plt.figure(figsize=(12, 4))
plt.plot(audio, color='gray', alpha=0.7, label='waveform')
for seg in speech_timestamps:
    plt.axvspan(seg['start'], seg['end'], alpha=0.3, color='green')

plt.title('VAD Result: Green = Speech')
plt.xlabel('Sample')
plt.legend(['Waveform', 'Speech'])
plt.show()

---
### 📝 まとめ

- Silero VAD は軽量で高精度、ONNX/TorchScript対応
- 無音・ノイズを正確に除去して後段のASR/TTS品質向上
- リアルタイム通信・音声認識の前処理として必須

🔗 [記事を読む](https://github.com/bonsai/sound-gen/blob/main/articles/親子対話：VAD.md)